# STEP 1: SINaS & GLOBI with GBIF backbone matching

In this script, we match downloaded Sinas and GLOBI data to make sure they both contain the same taxonomic names/synonyms

## 1. Setup and loading of SINAS data

In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import gzip
import re
import asyncio
import aiohttp
from tqdm.asyncio import tqdm_asyncio

In [2]:
# ==============================================================================
# --- CONFIG ---
# ==============================================================================
repo_root  = Path.cwd().parent
file_path  = repo_root / 'data' / 'SInAS_3.1.1.csv'
globi_path = repo_root / 'data' / 'interactions.tsv.gz'
 
GBIF_URL   = "https://api.gbif.org/v1/species/match"

In [8]:
# ==============================================================================
# 1. LOAD RAW DATA
# ==============================================================================
raw_data = pd.read_csv(file_path, sep=None, engine='python', quotechar='"')
print(f"✅ Data loaded. Rows: {len(raw_data)}")
 
species_col = 'taxon'
unique_sinas_names = raw_data[species_col].str.strip().dropna().unique().tolist()
print(f"Unique species to harmonize: {len(unique_sinas_names)}")

✅ Data loaded. Rows: 427956
Unique species to harmonize: 41171


## 2. Perform SINAS to GBIF backbone alignment

In [5]:
# ==============================================================================
# 2. ASYNC GBIF MATCHING (FULL TAXONOMIC HIERARCHY)
# ==============================================================================
async def match_one(
    session: aiohttp.ClientSession,
    name: str,
    semaphore: asyncio.Semaphore
) -> tuple[str, dict | None]:
    async with semaphore:
        for attempt in range(3):
            try:
                async with session.get(
                    GBIF_URL,
                    params={"name": name, "strict": "false"},
                    timeout=aiohttp.ClientTimeout(total=15),
                ) as resp:
                    if resp.status == 429:
                        await asyncio.sleep(2 ** attempt)
                        continue
                    resp.raise_for_status()
                    res = await resp.json()
 
                    if res.get("matchType") not in ("NONE", None):
                        return name, {
                            "canonical":   res.get("canonicalName"),
                            "gbif_id":     res.get("usageKey"),
                            # Full taxonomic hierarchy — all levels GBIF returns
                            "genus_id":    res.get("genusKey"),
                            "family_id":   res.get("familyKey"),
                            "order_id":    res.get("orderKey"),
                            "class_id":    res.get("classKey"),
                            "phylum_id":   res.get("phylumKey"),
                            "kingdom_id":  res.get("kingdomKey"),
                            # Matched rank and quality metrics
                            "match_type":  res.get("matchType"),
                            "confidence":  res.get("confidence"),
                            "rank":        res.get("rank"),
                        }
                    else:
                        return name, None
 
            except Exception as e:
                if attempt == 2:
                    print(f"\n  ⚠️  Failed '{name}' after 3 attempts: {e}")
                    return name, None
                await asyncio.sleep(1)
 
    return name, None
 
 
async def match_all(names: list[str], concurrency: int = 50) -> dict:
    semaphore = asyncio.Semaphore(concurrency)
    connector = aiohttp.TCPConnector(limit=concurrency)
 
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks   = [match_one(session, name, semaphore) for name in names]
        results = await tqdm_asyncio.gather(*tasks, desc="Matching names to GBIF backbone")
 
    return dict(results)

In [9]:
# ==============================================================================
# 3. MATCH SINAS NAMES
# ==============================================================================
# In a notebook:   sinas_map = await match_all(unique_sinas_names, concurrency=50)
# In a .py script: sinas_map = asyncio.run(match_all(unique_sinas_names, concurrency=50))
sinas_map = await match_all(unique_sinas_names, concurrency=50)
 
def get_field(name, field):
    entry = sinas_map.get(name)
    return entry[field] if entry else None
 
# Map all taxonomic levels back onto the raw dataframe
raw_data['gbif_canonical_name'] = raw_data[species_col].map(lambda x: get_field(x, 'canonical'))
raw_data['gbif_id']             = raw_data[species_col].map(lambda x: get_field(x, 'gbif_id'))
raw_data['gbif_genus_id']       = raw_data[species_col].map(lambda x: get_field(x, 'genus_id'))
raw_data['gbif_family_id']      = raw_data[species_col].map(lambda x: get_field(x, 'family_id'))
raw_data['gbif_order_id']       = raw_data[species_col].map(lambda x: get_field(x, 'order_id'))
raw_data['gbif_class_id']       = raw_data[species_col].map(lambda x: get_field(x, 'class_id'))
raw_data['gbif_phylum_id']      = raw_data[species_col].map(lambda x: get_field(x, 'phylum_id'))
raw_data['gbif_kingdom_id']     = raw_data[species_col].map(lambda x: get_field(x, 'kingdom_id'))
raw_data['gbif_match_type']     = raw_data[species_col].map(lambda x: get_field(x, 'match_type'))
raw_data['gbif_confidence']     = raw_data[species_col].map(lambda x: get_field(x, 'confidence'))
raw_data['gbif_rank']           = raw_data[species_col].map(lambda x: get_field(x, 'rank'))
 
matched = raw_data['gbif_id'].notna().sum()
print(f"✅ SINAS matching done. {matched}/{len(raw_data)} rows matched ({matched/len(raw_data):.1%})")
print(raw_data['gbif_match_type'].value_counts(dropna=False))
print(raw_data['gbif_rank'].value_counts(dropna=False))
 

Matching names to GBIF backbone: 100%|██████████| 41171/41171 [00:37<00:00, 1094.94it/s]


✅ SINAS matching done. 426805/427956 rows matched (99.7%)
gbif_match_type
EXACT         420632
HIGHERRANK      4806
FUZZY           1367
None            1151
Name: count, dtype: int64
gbif_rank
SPECIES       405807
SUBSPECIES      9926
VARIETY         6590
GENUS           3412
None            1151
FAMILY           562
FORM             366
PHYLUM           108
KINGDOM           16
ORDER             10
CLASS              8
Name: count, dtype: int64


## 3. Perform GLOBI to GBIF backbone alignment

In [10]:
# ==============================================================================
# 4. HARMONIZE GLOBI NAMES TO GBIF BACKBONE (FULL HIERARCHY)
# ==============================================================================
# globi_taxa: raw_name -> {"gbif_id": int, "rank": str, ...all levels...} | None
globi_taxa: dict[str, dict | None] = {}
 
with gzip.open(globi_path, 'rt', encoding='utf-8') as f:
    header     = f.readline().strip().split('\t')
    s_name_idx = header.index('sourceTaxonName')
    s_id_idx   = header.index('sourceTaxonId')
    t_name_idx = header.index('targetTaxonName')
    t_id_idx   = header.index('targetTaxonId')
 
    for line in tqdm(f, desc="Extracting GloBI taxa"):
        parts = line.strip().split('\t')
        for name_idx, id_idx in [(s_name_idx, s_id_idx), (t_name_idx, t_id_idx)]:
            if len(parts) <= id_idx:
                continue
            name   = parts[name_idx].strip()
            raw_id = parts[id_idx].strip()
            if not name or name in globi_taxa:
                continue
            m = re.search(r'GBIF:(\d+)', raw_id)
            # TSV-resolved: have gbif_id but no hierarchy yet — needs API pass for rank+hierarchy
            globi_taxa[name] = {"gbif_id": int(m.group(1)), "rank": None,
                                 "genus_id": None, "family_id": None,
                                 "order_id": None, "class_id": None,
                                 "phylum_id": None, "kingdom_id": None} if m else None
 
already_resolved = sum(1 for v in globi_taxa.values() if v is not None)
needs_lookup     = [n for n, v in globi_taxa.items() if v is None]
print(f"\nGloBI taxa total:        {len(globi_taxa)}")
print(f"  Resolved from TSV:     {already_resolved}")
print(f"  Needs GBIF API lookup: {len(needs_lookup)}")
 
# API lookup for names GloBI TSV didn't resolve at all
if needs_lookup:
    api_results = await match_all(needs_lookup, concurrency=50)
    for name, result in api_results.items():
        if result:
            globi_taxa[name] = {
                "gbif_id":    result['gbif_id'],
                "rank":       result['rank'],
                "genus_id":   result['genus_id'],
                "family_id":  result['family_id'],
                "order_id":   result['order_id'],
                "class_id":   result['class_id'],
                "phylum_id":  result['phylum_id'],
                "kingdom_id": result['kingdom_id'],
            }
        else:
            globi_taxa[name] = None
 
# Second pass: TSV-resolved entries have gbif_id but no hierarchy — fetch via API
needs_hierarchy = [n for n, v in globi_taxa.items() if v is not None and v['rank'] is None]
print(f"  Fetching full hierarchy for TSV-resolved entries: {len(needs_hierarchy)}")
 
if needs_hierarchy:
    hierarchy_results = await match_all(needs_hierarchy, concurrency=50)
    for name, result in hierarchy_results.items():
        if result and globi_taxa.get(name):
            globi_taxa[name].update({
                "rank":       result['rank'],
                "genus_id":   result['genus_id'],
                "family_id":  result['family_id'],
                "order_id":   result['order_id'],
                "class_id":   result['class_id'],
                "phylum_id":  result['phylum_id'],
                "kingdom_id": result['kingdom_id'],
            })
 
resolved = sum(1 for v in globi_taxa.values() if v is not None)
print(f"✅ GloBI harmonization done. {resolved}/{len(globi_taxa)} names resolved to GBIF IDs")
 
# Rank distribution in GloBI
from collections import Counter
rank_counts = Counter(
    v['rank'] for v in globi_taxa.values()
    if v is not None and v.get('rank')
)
print("GloBI rank distribution:", dict(rank_counts))

Extracting GloBI taxa: 20361182it [09:37, 35241.40it/s]



GloBI taxa total:        1040927
  Resolved from TSV:     122767
  Needs GBIF API lookup: 918160


Matching names to GBIF backbone: 100%|██████████| 918160/918160 [12:58<00:00, 1179.54it/s]


  Fetching full hierarchy for TSV-resolved entries: 122767


Matching names to GBIF backbone: 100%|██████████| 122767/122767 [01:48<00:00, 1132.44it/s]


✅ GloBI harmonization done. 497445/1040927 names resolved to GBIF IDs
GloBI rank distribution: {'SPECIES': 372821, 'GENUS': 71753, 'CLASS': 465, 'FAMILY': 5017, 'SUBSPECIES': 14418, 'ORDER': 770, 'KINGDOM': 1797, 'PHYLUM': 1262, 'VARIETY': 7524, 'FORM': 297, 'UNRANKED': 20889}


## 4. Verify overlap between Sinas & Globi resolution

In [11]:
# ==============================================================================
# 5. COMPUTE OVERLAP (SINAS ∩ GLOBI)
# ==============================================================================
resolved_globi_ids = {v['gbif_id'] for v in globi_taxa.values() if v is not None}
sinas_ids          = set(raw_data['gbif_id'].dropna().unique())
overlap            = sinas_ids.intersection(resolved_globi_ids)
 
print(f"\nSInAS Species:           {len(sinas_ids)}")
print(f"Interactions found for:  {len(overlap)}")
print(f"Actual Useful Coverage:  {(len(overlap)/len(sinas_ids))*100:.2f}%")


SInAS Species:           38586
Interactions found for:  26541
Actual Useful Coverage:  68.78%


## 5. Save the matched Sinas & Globi datasets

In [12]:
# ==============================================================================
# 6. SAVE FILTERED SINAS SUBSET
# ==============================================================================
sinas_subset = raw_data[raw_data['gbif_id'].isin(overlap)].copy()
sinas_subset.to_csv(repo_root / 'data' / 'sinas_matched_species.csv', index=False)
print(f"\n📁 Saved {len(sinas_subset)} matched SInAS rows → 'sinas_matched_species.csv'")
 
 
# ==============================================================================
# 7. SAVE ID BRIDGE TABLE
# ==============================================================================
globi_mapping_df = pd.DataFrame([
    {'globi_taxon_name': k, 'gbif_id': v['gbif_id'], 'gbif_rank': v['rank']}
    for k, v in globi_taxa.items()
    if v is not None and v['gbif_id'] in overlap
])
 
sinas_names_df = raw_data[['taxon', 'gbif_id']].dropna().drop_duplicates()
id_bridge_df   = pd.merge(sinas_names_df, globi_mapping_df, on='gbif_id', how='inner')
id_bridge_df.rename(columns={'taxon': 'sinas_taxon_name'}, inplace=True)
 
id_bridge_df.to_csv(repo_root / 'data' / 'sinas_globi_id_bridge.csv', index=False)
print(f"📁 Saved ID bridge table → 'sinas_globi_id_bridge.csv'")
 
 
# ==============================================================================
# 8. STREAM GLOBI TSV AND EXTRACT INTERACTION NETWORK (WITH FULL HIERARCHY)
# ==============================================================================
matched_network = []
 
with gzip.open(globi_path, 'rt', encoding='utf-8') as f:
    header          = f.readline().strip().split('\t')
    s_name_idx      = header.index('sourceTaxonName')
    t_name_idx      = header.index('targetTaxonName')
    interaction_idx = header.index('interactionTypeName')
    path_idx        = header.index('referenceCitation') if 'referenceCitation' in header else None
 
    for line in tqdm(f, desc="Extracting matched GloBI network edges"):
        parts = line.strip().split('\t')
        if len(parts) <= max(s_name_idx, t_name_idx):
            continue
 
        s_name  = parts[s_name_idx].strip()
        t_name  = parts[t_name_idx].strip()
 
        s_entry = globi_taxa.get(s_name)
        t_entry = globi_taxa.get(t_name)
 
        s_gbif_id = s_entry['gbif_id']   if s_entry else None
        t_gbif_id = t_entry['gbif_id']   if t_entry else None
        s_rank    = s_entry['rank']       if s_entry else None
        t_rank    = t_entry['rank']       if t_entry else None
 
        if s_gbif_id in overlap or t_gbif_id in overlap:
            matched_network.append({
                'source_taxon_name': s_name,
                'source_gbif_id':    s_gbif_id,
                'source_rank':       s_rank,
                'interaction_type':  parts[interaction_idx] if interaction_idx < len(parts) else None,
                'target_taxon_name': t_name,
                'target_gbif_id':    t_gbif_id,
                'target_rank':       t_rank,
                'reference':         parts[path_idx] if path_idx and path_idx < len(parts) else None,
            })
 
globi_network_df = pd.DataFrame(matched_network)
 
# Deduplicate: same source, target, and interaction type is a duplicate
globi_network_df.drop_duplicates(
    subset=['source_gbif_id', 'target_gbif_id', 'interaction_type'],
    inplace=True
)
 
print(f"\nTotal unique edges after deduplication: {len(globi_network_df)}")
print("Rank combinations in network:")
print(
    globi_network_df
    .groupby(['source_rank', 'target_rank'], dropna=False)
    .size()
    .sort_values(ascending=False)
    .head(15)
)
 
 
# ==============================================================================
# 9. SPLIT INTO SPECIES-LEVEL AND HIGHER-ORDER INTERACTION FILES
# ==============================================================================
species_mask = (
    (globi_network_df['source_rank'] == 'SPECIES') &
    (globi_network_df['target_rank'] == 'SPECIES')
)
 
species_interactions      = globi_network_df[species_mask].copy()
higher_order_interactions = globi_network_df[~species_mask].copy()
 
species_interactions.to_csv(
    repo_root / 'data' / 'matched_globi_network.csv', index=False
)
higher_order_interactions.to_csv(
    repo_root / 'data' / 'matched_globi_network_higher_order.csv', index=False
)
 
print(f"\n🚀 SUCCESS!")
print(f"  Species-level edges:      {len(species_interactions)}")
print(f"  Higher-order edges:       {len(higher_order_interactions)}")
print(f"  Saved → 'matched_globi_network.csv'")
print(f"  Saved → 'matched_globi_network_higher_order.csv'")


📁 Saved 378728 matched SInAS rows → 'sinas_matched_species.csv'
📁 Saved ID bridge table → 'sinas_globi_id_bridge.csv'


Extracting matched GloBI network edges: 20361182it [10:59, 30865.61it/s]



Total unique edges after deduplication: 1396853
Rank combinations in network:
source_rank  target_rank
SPECIES      SPECIES        952088
             GENUS          128480
GENUS        SPECIES        101034
SPECIES      FAMILY          33950
             NaN             23285
             KINGDOM         18035
NaN          SPECIES         15558
SPECIES      SUBSPECIES      15493
SUBSPECIES   SPECIES         13919
GENUS        GENUS           11454
FAMILY       SPECIES          9778
UNRANKED     SPECIES          8934
SPECIES      VARIETY          8277
             CLASS            7794
             ORDER            5818
dtype: int64

🚀 SUCCESS!
  Species-level edges:      952088
  Higher-order edges:       444765
  Saved → 'matched_globi_network.csv'
  Saved → 'matched_globi_network_higher_order.csv'
